# Лекция: Построение матрицы корреляции в Python

**Дисциплина:** Введение в анализ больших данных  
**Задание 6** (адаптация с языка R на Python)

В задании используется датасет **NMES1988**. Основные задачи:
- проверка нормальности (Shapiro–Wilk)
- корреляция Спирмена
- матрица корреляций + визуализация
- проверка значимости коэффициентов
- корреляции по группам (пол, регион)

Библиотеки: **pandas**, **numpy**, **scipy.stats**, **seaborn**, **matplotlib**.


## 0. Импорт и загрузка данных


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

print("Библиотеки загружены")


In [ ]:
url = "https://vincentarelbundock.github.io/Rdatasets/csv/AER/NMES1988.csv"
df = pd.read_csv(url)

cols = ["visits", "health", "chronic", "adl", "region", "age",
        "gender", "married", "school", "income", "employed", "insurance"]
available = [c for c in cols if c in df.columns]
nmes = df[available].copy()

print("Размерность:", nmes.shape)
print("Столбцы:", nmes.columns.tolist())
nmes.head()


### Количественные переменные


In [ ]:
quant_cols = ["visits", "chronic", "age", "school", "income"]
print("Количественные показатели:", quant_cols)
nmes[quant_cols].describe()


---
## 1. Проверка нормальности (Shapiro–Wilk)

В R: `shapiro.test(x)`

В Python: `scipy.stats.shapiro(x)`

**Важно:** Shapiro–Wilk надёжен при n ≤ ~5000. При больших n часто отклоняет H0 даже при небольших отклонениях от нормальности.


In [ ]:
print("=== Shapiro–Wilk test ===")
print("H0: данные распределены нормально\n")

for col in quant_cols:
    x = nmes[col].dropna()
    if len(x) > 5000:
        x = x.sample(5000, random_state=42)
    stat, p = stats.shapiro(x)
    verdict = "нормальное" if p > 0.05 else "НЕ нормальное"
    print(f"{col:10s}: W = {stat:.4f}, p = {p:.4e}  →  {verdict}")


Большинство счётных и социально-экономических переменных **не нормальны**, поэтому для корреляций предпочтительнее **Спирмен** (ранговый метод).


---
## 2. Корреляция Спирмена: visits и age

В R: `cor.test(x, y, method = "spearman")`


In [ ]:
r, p = stats.spearmanr(nmes["visits"], nmes["age"], nan_policy="omit")

print("Корреляция Спирмена: visits ~ age")
print(f"  rho     = {r:.4f}")
print(f"  p-value = {p:.4e}")

if p < 0.05:
    print("  → связь статистически значима")
else:
    print("  → связь НЕ значима")


---
## 3. Матрица корреляций для количественных показателей

В R: `cor(data, method = "spearman")` + `corrplot()`


In [ ]:
corr_matrix = nmes[quant_cols].corr(method="spearman")
print("Матрица корреляций (Спирмен):")
print(corr_matrix.round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(corr_matrix, annot=False, cmap="RdYlBu_r", center=0,
            vmin=-1, vmax=1, square=True, ax=axes[0],
            cbar_kws={"shrink": 0.8})
axes[0].set_title("Корреляционная матрица (Спирмен)")

sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0,
            vmin=-1, vmax=1, square=True, ax=axes[1],
            cbar_kws={"shrink": 0.8})
axes[1].set_title("Корреляционная матрица + значения")

plt.tight_layout()
plt.show()


---
## 4. Проверка значимости коэффициентов корреляции

В R: `corr.test()` из пакета **psych**

В Python можно посчитать p-value для каждой пары через `scipy.stats.spearmanr`.
Сфокусируемся на коэффициентах **|r| < 0.5** (как в задании).


In [ ]:
def correlation_pvalues(df, method="spearman"):
    """Матрица p-value для всех пар столбцов"""
    cols = df.columns
    n = len(cols)
    pvals = pd.DataFrame(np.ones((n, n)), index=cols, columns=cols)
    for i in range(n):
        for j in range(i + 1, n):
            if method == "spearman":
                _, p = stats.spearmanr(df.iloc[:, i], df.iloc[:, j], nan_policy="omit")
            else:
                _, p = stats.pearsonr(df.iloc[:, i].dropna(), df.iloc[:, j].dropna())
            pvals.iloc[i, j] = p
            pvals.iloc[j, i] = p
    return pvals

pvals = correlation_pvalues(nmes[quant_cols], method="spearman")
print("Матрица p-value (Спирмен):")
print(pvals.round(4))


In [ ]:
print("=== Пары с |r| < 0.5 ===\n")
for i, c1 in enumerate(quant_cols):
    for c2 in quant_cols[i+1:]:
        r = corr_matrix.loc[c1, c2]
        p = pvals.loc[c1, c2]
        if abs(r) < 0.5:
            sig = "значима" if p < 0.05 else "НЕ значима"
            print(f"{c1:8s} – {c2:8s}:  r = {r:7.3f},  p = {p:.4e}  →  {sig}")


### Анализ отдельно для мужчин и женщин


In [ ]:
print("Уровни gender:", nmes["gender"].unique())

for g in nmes["gender"].dropna().unique():
    subset = nmes.loc[nmes["gender"] == g, quant_cols]
    corr_g = subset.corr(method="spearman")
    pvals_g = correlation_pvalues(subset, method="spearman")

    print(f"\n========== gender = {g} (n = {len(subset)}) ==========")
    print(corr_g.round(3))

    print(f"\nПары с |r| < 0.5:")
    for i, c1 in enumerate(quant_cols):
        for c2 in quant_cols[i+1:]:
            r = corr_g.loc[c1, c2]
            p = pvals_g.loc[c1, c2]
            if abs(r) < 0.5:
                sig = "*" if p < 0.05 else ""
                print(f"  {c1:8s}–{c2:8s}: r={r:7.3f}, p={p:.3e} {sig}")


In [ ]:
genders = nmes["gender"].dropna().unique()
fig, axes = plt.subplots(1, len(genders), figsize=(6 * len(genders), 5))

if len(genders) == 1:
    axes = [axes]

for ax, g in zip(axes, genders):
    subset = nmes.loc[nmes["gender"] == g, quant_cols]
    corr_g = subset.corr(method="spearman")
    sns.heatmap(corr_g, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0,
                vmin=-1, vmax=1, square=True, ax=ax)
    ax.set_title(f"Спирмен, gender = {g}")

plt.tight_layout()
plt.show()


---
## 5. Корреляция visits с другими показателями по регионам

Группируем по `region` и считаем корреляции visits с chronic, age, school, income.


In [ ]:
other_quant = ["chronic", "age", "school", "income"]
regions = nmes["region"].dropna().unique()

results = []
for reg in regions:
    subset = nmes.loc[nmes["region"] == reg]
    for col in other_quant:
        r, p = stats.spearmanr(subset["visits"], subset[col], nan_policy="omit")
        results.append({
            "region": reg,
            "variable": col,
            "rho": r,
            "p_value": p,
            "n": len(subset)
        })

res_df = pd.DataFrame(results)
print(res_df.round(4).to_string(index=False))


In [ ]:
pivot = res_df.pivot(index="region", columns="variable", values="rho")

plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlBu_r", center=0,
            vmin=-0.5, vmax=0.5)
plt.title("Корреляция Спирмена: visits ~ другие показатели\nпо регионам")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(other_quant))
width = 0.2

for i, reg in enumerate(sorted(regions)):
    vals = res_df.loc[res_df["region"] == reg].set_index("variable").loc[other_quant, "rho"]
    ax.bar(x + i * width, vals, width, label=str(reg))

ax.set_xticks(x + width * (len(regions) - 1) / 2)
ax.set_xticklabels(other_quant)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Spearman rho")
ax.set_title("Корреляция visits с показателями по регионам")
ax.legend(title="region")
plt.tight_layout()
plt.show()


---
## Краткие выводы (шаблон для анализа)

1. **Нормальность:** количественные переменные, как правило, не нормальны → используем Спирмена.
2. **visits ~ age:** смотрите на rho и p-value — есть ли статистически значимая монотонная связь.
3. **Матрица корреляций:** какие пары наиболее сильно связаны (|r| ближе к 1).
4. **По полу:** отличаются ли структура и сила связей у мужчин и женщин.
5. **По регионам:** стабильна ли связь visits с age/income/school/chronic в разных регионах.

---
## Шпаргалка: R → Python

| Задача в R | Python |
|------------|--------|
| `shapiro.test(x)` | `stats.shapiro(x)` |
| `cor.test(x, y, method="spearman")` | `stats.spearmanr(x, y)` |
| `cor(df, method="spearman")` | `df.corr(method="spearman")` |
| `corrplot(...)` | `sns.heatmap(corr, annot=True, ...)` |
| `corr.test()` (psych) | цикл по `stats.spearmanr` / своя функция p-value |
| группировка | `df.groupby(...)` + корреляции внутри групп |

---
## Рекомендации

1. При n > 5000 Shapiro–Wilk почти всегда даёт p < 0.05 — смотрите также Q–Q plot и гистограммы.
2. Для ранговых корреляций используйте `nan_policy="omit"`.
3. Значимость при большом n легко «ловятся» даже слабые связи — интерпретируйте и величину r.
4. Документация: [scipy.stats.spearmanr](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html), [seaborn.heatmap](https://seaborn.pydata.org/generated/seaborn.heatmap.html).

**Удачи с выполнением Задания 6!**
